In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2

# Trick to plot with tex
os.environ["LD_LIBRARY_PATH"] = ""
os.environ["CONDA_PREFIX"] = "/home/guerrini/.conda/envs/sp_validation_3.11"

sys.path.append("/home/guerrini/sp_validation/cosmo_inference/scripts/")

from getdist import plots, loadMCSamples
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import chain_postprocessing as cp

plt.style.use("/home/guerrini/matplotlib_config/paper.mplstyle")

plt.rc("text", usetex=True)

sns.set_palette("husl")

g = plots.get_subplot_plotter(width_inch=30)
g.settings.axes_fontsize = 60
g.settings.axes_labelsize = 60
g.settings.alpha_filled_add = 0.7
g.settings.legend_fontsize = 60

%matplotlib inline

# SPECIFY DATA DIRECTORY AND DESIRED CHAINS TO ANALYSE
root_dir = "/n09data/guerrini/output_chains/"
root_external = "/n09data/guerrini/output_chains/ext_data/"
blind = "B"

colour_blind = {"A": "royalblue", "B": "crimson", "C": "forestgreen"}

roots = [
    f"SP_v1.4.6.3_leak_corr_{blind}",
    f"SP_v1.4.6.3_{blind}_fiducial_config",
    "Planck18",
    "DES_Y3",
    "DES_Y3_cell",
    "KiDS-Legacy_xipm",
    "KiDS-Legacy_bandpowers",
    "KiDS-Legacy_cosebis",
    "DES+KiDS",
    "HSC_Y3",
    "HSC_Y3_cell",
    f"SP_v1.4.6.3_leak_corr_kmax=5Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_kmax=3Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_kmax=1Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_include_large_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_small_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_large_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_halofit_{blind}",
    f"SP_v1.4.6.3_leak_corr_HMCode_nobar_{blind}",
    f"SP_v1.4.6.3_leak_corr_OneCov_{blind}",
    f"SP_v1.4.6.3_{blind}",
]

legend_labels = [
    r"UNIONS $C_\ell$ (This work)",
    r"UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026)",
    r"\textit{Planck} 2018",
    r"DES Y3 $\xi_\pm(\vartheta)$",
    r"DES Y3 $C_\ell$",
    r"KiDS-Legacy $\xi_\pm(\vartheta)$",
    r"KiDS-Legacy $C_E$",
    r"KiDS-Legacy $E_n$",
    r"DES Y3 + KiDS-1000 combined",
    r"HSC Y3 $\xi_\pm(\vartheta)$",
    r"HSC Y3 $C_\ell$",
    r"$k_{\rm max}=5h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=2048$",
    r"$k_{\rm max}=3h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=1800$",
    r"$k_{\rm max}=1h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=500$",
    r"Include large scales, $\ell_{\rm max}=1600$",
    r"Small scales only",
    r"Large scales only",
    r"\texttt{Halofit}",
    r"\texttt{HMCode} no baryons",
    r"\texttt{OneCovariance} only",
    r"No leakage correction",
]

colours = [
    colour_blind["B"],
    "darkorange",
    "violet",
    "black",
    "black",
    "black",
    "black",
    "black",
    "black",
    "black",
    "black",
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
]

categories = [
    "harmonic",
    "configuration",
    "external",
    "external",
    "external",
    "external_compute_sample",
    "external_compute_sample",
    "external_compute_sample",
    "external",
    "external",
    "external_compute_sample",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
]

# Add the blinds to each list
for bl in ["A", "B", "C"]:
    if bl != blind:
        roots.append(f"SP_v1.4.6.3_leak_corr_{bl}")
        legend_labels.append(rf"UNIONS $C_\ell$, Blind {bl}")
        colours.append(colour_blind[bl])
        categories.append("harmonic")

print(roots)

In [ ]:
chains = []
for i, root in enumerate(roots):
    category = categories[i]
    if category != "external":
        if category == "configuration":
            path_samples = os.path.join(root_dir, f"{root}/samples_{root}.txt")
            path_getdist = os.path.join(root_dir, f"{root}/getdist_{root}")
        elif category == "harmonic":
            path_samples = os.path.join(
                root_dir, f"{root}/{root}/samples_{root}_cell.txt"
            )
            path_getdist = os.path.join(root_dir, f"{root}/{root}/getdist_{root}")
        elif category == "external_compute_sample":
            path_samples = os.path.join(root_dir, f"ext_data/{root}/samples_{root}.txt")
            path_getdist = os.path.join(root_dir, f"ext_data/{root}/getdist_{root}")
        else:
            raise ValueError(f"The category, {category}, of {root} is not correct")

        if category == "external_compute_sample" and "Legacy" in root:
            chain_type = "nautilus"
        else:
            chain_type = "polychord"
        cp.load_samples_and_write_paramnames(
            path_samples, path_getdist + ".paramnames", chain_type=chain_type
        )
        cp.write_samples_getdist_format(
            path_samples, path_getdist + ".txt", chain_type=chain_type
        )
        chains.append(cp.load_chain(path_getdist, smoothing_scale=0.5))
    else:
        path_getdist = os.path.join(root_dir, f"ext_data/{root}/getdist_{root}")
        chains.append(cp.load_chain(path_getdist))

In [ ]:
name_list = [
    "OMEGA_M",
    "ombh2",
    "h0",
    "n_s",
    "SIGMA_8",
    "S_8",
    "s_8_input",
    "logt_agn",
    "a",
    "m1",
    "bias_1",
]
label_list = [
    r"\Omega_{\rm m}",
    r"\omega_b h^2",
    r"h_0",
    r"n_s",
    r"\sigma_8",
    r"S_8",
    r"S_8",
    r"\log T_{\rm AGN}",
    r"A_{\rm IA}",
    r"m_1",
    r"\Delta z_1",
]

for i, chain in enumerate(chains):
    print(legend_labels[i])
    param_names = chain.getParamNames()
    for name, label in zip(name_list, label_list):
        try:
            param_names.parWithName(name).label = label
        except:
            warnings.warn(f"Parameter {name} not found in chain {roots[i]}.")

In [ ]:
# Micro management of external chains
# Account for the missing parameter conventions
# OMEGA_M not in DES_Y3_cell
idx = roots.index("DES_Y3_cell")
cp.adjust_paramname_chain(chains[idx], "omega_m", "OMEGA_M", r"\Omega_{\rm m}")
cp.derive_parameter_S8(chains[idx])

# OMEGA_M not in KiDS-1000
try:
    idx = roots.index("KiDS-1000")
    cp.adjust_paramname_chain(chains[idx], "omega_m", "OMEGA_M", r"\Omega_{\rm m}")
except:
    print("KiDS-1000 chain not found, skipping parameter name adjustment.")

# S8 to derive in KiDS-Legacy chains
try:
    idx = roots.index("KiDS-Legacy_xipm")
    cp.derive_parameter_S8(chains[idx])
except:
    print("KiDS-Legacy_xipm chain not found, skipping S8 derivation.")
try:
    idx = roots.index("KiDS-Legacy_bandpowers")
    cp.derive_parameter_S8(chains[idx])
except:
    print("KiDS-Legacy_bandpowers chain not found, skipping S8 derivation.")
try:
    idx = roots.index("KiDS-Legacy_cosebis")
    cp.derive_parameter_S8(chains[idx])
except:
    print("KiDS-Legacy_cosebis chain not found, skipping S8 derivation.")

# OMEGA_M not in DES+KiDS
idx = roots.index("DES+KiDS")
cp.adjust_paramname_chain(chains[idx], "omega_m", "OMEGA_M", r"\Omega_{\rm m}")

# OMEGA_M not in HSC_Y3_cell
idx = roots.index("HSC_Y3_cell")
cp.adjust_paramname_chain(chains[idx], "omega_m", "OMEGA_M", r"\Omega_{\rm m}")

In [ ]:
best_fit_method = "2Dkde"

param_values = np.array(
    [
        "# Expt",
        "Colour",
        "S8_Mean",
        "S8_low",
        "S8_high",
        "sigma_8_Mean",
        "sigma_8_low",
        "sigma_8_high",
        "Omega_m_Mean",
        "Omega_m_low",
        "Omega_m_high",
    ]
)
escaped = np.char.replace(legend_labels, "\\", "\\\\")
for i, chain in enumerate(chains):
    print(chain.root)
    margestats = chain.getMargeStats()
    likestats = chain.getLikeStats()

    s8_stats = margestats.parWithName("S_8")
    sigma8_stats = margestats.parWithName("SIGMA_8")
    omegam_stats = margestats.parWithName("OMEGA_M")

    best_fit = cp.extract_best_fit_params(chain, best_fit_method=best_fit_method)

    param_values = np.vstack(
        (
            param_values,
            [
                escaped[i],
                colours[i],
                best_fit["S_8"],
                best_fit["S_8"] - s8_stats.limits[0].lower,
                s8_stats.limits[0].upper - best_fit["S_8"],
                best_fit["SIGMA_8"],
                best_fit["SIGMA_8"] - sigma8_stats.limits[0].lower,
                sigma8_stats.limits[0].upper - best_fit["SIGMA_8"],
                best_fit["OMEGA_M"],
                best_fit["OMEGA_M"] - omegam_stats.limits[0].lower,
                omegam_stats.limits[0].upper - best_fit["OMEGA_M"],
            ],
        )
    )
print(param_values)
np.savetxt(
    f"{root_dir}/param_values.txt",
    param_values,
    fmt=["%s" for i in range(11)],
    delimiter=";",
)

In [ ]:
# Load the value of the parameters
cosmo = np.loadtxt(
    f"{root_dir}/param_values.txt",
    dtype={
        "names": (
            "Expt",
            "colour",
            "s8_mean",
            "s8_low",
            "s8_high",
            "sigma8_mean",
            "sigma8_low",
            "sigma8_high",
            "omegam_mean",
            "omegam_low",
            "omegam_high",
        ),
        "formats": (
            "U250",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
            "U20",
        ),
    },
    skiprows=1,
    delimiter=";",
)
expt = np.char.replace(cosmo["Expt"], "\\\\", "\\")
colours = cosmo["colour"]
s8_mean = cosmo["s8_mean"].astype(np.float64)
s8_low = cosmo["s8_low"].astype(np.float64)
s8_high = cosmo["s8_high"].astype(np.float64)
sigma8_mean = cosmo["sigma8_mean"].astype(np.float64)
sigma8_low = cosmo["sigma8_low"].astype(np.float64)
sigma8_high = cosmo["sigma8_high"].astype(np.float64)
omegam_mean = cosmo["omegam_mean"].astype(np.float64)
omegam_low = cosmo["omegam_low"].astype(np.float64)
omegam_high = cosmo["omegam_high"].astype(np.float64)

In [ ]:
def get_sigma_tension(mean1, low1, high1, mean2, low2, high2):
    sigma1 = 0.5 * (high1 + low1)
    sigma2 = 0.5 * (high2 + low2)
    delta_mean = np.abs(mean1 - mean2)
    sigma_tension = delta_mean / np.sqrt(sigma1**2 + sigma2**2)
    sign = 1 if mean1 > mean2 else -1
    return sigma_tension * sign

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(10, 6))
gs = GridSpec(1, 3, width_ratios=[1, 0.5, 0.5])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharey=ax1)
ax3 = fig.add_subplot(gs[2], sharey=ax1)

axs = [ax1, ax2, ax3]

params = [
    (s8_mean, s8_low, s8_high, r"$S_8$"),
    (sigma8_mean, sigma8_low, sigma8_high, r"$\sigma_8$"),
    (omegam_mean, omegam_low, omegam_high, r"$\Omega_{\rm m}$"),
]
reference = r"UNIONS $C_\ell$ (This work)"
separation_after = [
    r"UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026)",
    r"HSC Y3 $C_\ell$",
    r"Large scales only",
    r"\texttt{HMCode} no baryons",
    r"\texttt{OneCovariance} only",
    r"No leakage correction",
]
list_section_index = [r"(ii)", r"(iii)", r"(iv)", r"(v)", r"(vi)", r"(vii)"]

preliminary_watermark = False
blind_axes = False
row_spacing = 0.1

index_ref = np.where(expt == reference)[0][0]

y = np.arange(len(expt))
for ax, param in zip(axs, params):
    means, lows, highs, label = param
    for i, mean, low, high, color in zip(y, means, lows, highs, colours):
        ax.errorbar(
            mean,
            0.05 + i * row_spacing,
            xerr=np.array([low, high])[:, None],
            fmt="o",
            color=color,
            ecolor=color,
            elinewidth=2,
            capsize=3,
        )
    ax.set_xlabel(label, fontsize=14)

    ax.grid(False)
    ax.tick_params(axis="y", left=False, labelleft=False)
    if label == r"$S_8$":
        ax.axvspan(
            s8_mean[index_ref] - s8_low[index_ref],
            s8_mean[index_ref] + s8_high[index_ref],
            color=colours[index_ref],
            alpha=0.2,
        )
        ax.set_xlim(0.20, 1.1)
        if blind_axes:
            ref_tick = np.mean(s8_mean[:4])
            ax.set_xticks([ref_tick + i * 0.1 for i in range(-5, 5)], labels=[])
        else:
            ax.set_xticks([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    elif label == r"$\sigma_8$":
        ax.axvspan(
            sigma8_mean[index_ref] - sigma8_low[index_ref],
            sigma8_mean[index_ref] + sigma8_high[index_ref],
            color=colours[index_ref],
            alpha=0.2,
        )
        ax.set_xlim(0.5, 1.2)
        if blind_axes:
            ref_tick = np.mean(sigma8_mean[:4])
            ax.set_xticks([ref_tick + i * 0.2 for i in range(-2, 2)], labels=[])
    elif label == r"$\Omega_{\rm m}$":
        ax.axvspan(
            omegam_mean[index_ref] - omegam_low[index_ref],
            omegam_mean[index_ref] + omegam_high[index_ref],
            color=colours[index_ref],
            alpha=0.2,
        )
        ax.set_xlim(0.1, 0.5)
        if blind_axes:
            ref_tick = np.mean(omegam_mean[:4])
            ax.set_xticks([ref_tick + i * 0.1 for i in range(-2, 3)], labels=[])


axs[0].set_yticks(0.05 + y * row_spacing)
axs[0].set_yticklabels([])
for label, color in zip(expt, colours):
    axs[0].text(
        0.21,
        0.05 + row_spacing * np.where(expt == label)[0][0],
        label,
        fontsize=12,
        ha="left",
        va="center",
        color=color,
    )
    if label != reference:
        index = np.where(expt == label)[0][0]
        s8_tension = get_sigma_tension(
            s8_mean[index],
            s8_low[index],
            s8_high[index],
            s8_mean[index_ref],
            s8_low[index_ref],
            s8_high[index_ref],
        )
        sign_str = "+" if s8_tension > 0 else "-"
        axs[0].text(
            1.0,
            0.05 + row_spacing * index,
            rf"${sign_str}{np.abs(s8_tension):.2f}" + r"\, \sigma$",
            fontsize=10,
            ha="left",
            va="center",
            color=color,
        )
# Add separation lines
for i, sep in enumerate(separation_after):
    index_sep = np.where(expt == sep)[0][0]
    for ax in axs:
        ax.axhline(
            row_spacing * (index_sep + 0.95),
            color="black",
            linestyle="dotted",
            linewidth=1,
        )
        axs[0].text(
            0.20,
            0.05 + row_spacing * (index_sep + 1),
            list_section_index[i],
            fontsize=14,
            fontweight="bold",
            va="center",
            ha="right",
        )


# --- Add section labels (i), (ii)) ---
axs[0].text(0.20, 0.05, r"(i)", fontsize=14, fontweight="bold", va="center", ha="right")

if preliminary_watermark:
    plt.figtext(
        0.5,
        0.5,
        "PRELIMINARY",
        fontsize=50,
        color="gray",
        ha="center",
        va="center",
        alpha=0.3,
        rotation=330,
    )

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig("./plots/whisker_plot.png", dpi=300)
# Save pdf
plt.savefig("./plots/whisker_plot.pdf")
plt.show()